# Statewide event plotting QPE only
- The QPE only version (i.e. no model preds) analigous to what is in : /home/csutter/DRIVE-clean/weather_events/notebooks/statewide_event_gif.ipynb

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import imageio.v2 as imageio 
import os
import requests
import gzip
import xarray as xr
from glob import glob
import numpy as np

# --- 1. SET UP A DARK MODE RADAR STYLE ---
plt.style.use('dark_background')
plt.rcParams['font.family'] = 'serif'

# --- 2. GENERATE TIMELINE ---
# Automatically creates a list of timestamps every 4 hours
timeline = pd.date_range(start='2026-01-23 00:00', end='2026-01-29 23:59', freq='4h')

def get_mrms_s3(ts, product_type="PrecipRate", worker_index=999):
    """Downloads and subsets MRMS data from S3 using pure xarray."""
    date_str = ts.strftime('%Y%m%d')
    time_str = ts.strftime('%H%M') + "00"
    
    base_url = f"https://noaa-mrms-pds.s3.amazonaws.com/CONUS/{product_type}_00.00"
    file_name = f"MRMS_{product_type}_00.00_{date_str}-{time_str}.grib2.gz"
    full_url = f"{base_url}/{date_str}/{file_name}"
    
    r = requests.get(full_url)

    if r.status_code == 200:
        temp_grib = f"temp_{product_type}_{ts.strftime('%Y%m%d%H%M%S')}_idx{worker_index}.grib2" 
        
        with open(temp_grib, "wb") as f:
            f.write(gzip.decompress(r.content))

        ds_full = xr.open_dataset(temp_grib, engine="cfgrib", backend_kwargs={'indexpath': ''})
        ds = ds_full.sel(latitude=slice(47.5, 38.5), longitude=slice(278.0, 291.0)).load()
        
        ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
        ds = ds.sortby('longitude') 
        
        ds_full.close()
        del ds_full
        
        if os.path.exists(temp_grib):
            os.remove(temp_grib)
            for f in glob(f"{temp_grib}*.idx"):
                os.remove(f)
        
        return ds
    else:
        return None


frame_filenames = []
print(f"Generating {len(timeline)} map frames...")

# --- 3. THE ANIMATION LOOP ---
for i, ts in enumerate(timeline):
    
    # MRMS requires 2-minute even intervals (though on the 4-hour mark it's already :00)
    even_min = (ts.minute // 2) * 2
    mrms_ts = ts.replace(minute=even_min, second=0)
    
    print(f"Processing {mrms_ts}...")
    qpe_ds = get_mrms_s3(mrms_ts, "PrecipRate", i)
    
    fig, ax = plt.subplots(figsize=(14, 10))
    
    if qpe_ds is not None:
        var_name = list(qpe_ds.data_vars)[0] 
        da = qpe_ds[var_name]
        
        # Filter out absolute 0s so the background stays dark
        da = da.where(da > 0.05)
        
        # USE CONTOURF FOR SMOOTH GRADIENTS
        # levels=20 creates 20 distinct color bands for a very smooth transition
        plot = da.plot.contourf(
            ax=ax, 
            x='longitude', 
            y='latitude', 
            levels=np.linspace(0.1, 10.0, 20), 
            cmap='YlGnBu_r',   # Yellow-Green-Blue (reversed) looks amazing on dark mode!
            add_colorbar=False, # We will add a custom one below
            zorder=2
        )
        
        # Add a sleek colorbar at the bottom
        cbar = plt.colorbar(plot, ax=ax, orientation='horizontal', pad=0.05, aspect=50)
        cbar.set_label('Precipitation Rate (mm/hr)', fontsize=12, color='white')
        cbar.ax.tick_params(colors='white')
        
    else:
        # If the S3 bucket is missing data for this specific 4-hour block, keep the frame but add text
        ax.text(0.5, 0.5, 'No MRMS Data Available', color='gray', fontsize=20, 
                ha='center', va='center', transform=ax.transAxes)

    # --- FORMATTING (No Geopandas needed!) ---
    # The Magic Math Trick to fix map stretching
    ax.set_aspect(1.0 / np.cos(np.radians(43.0)))
    
    # NY State Bounding Box
    ax.set_xlim(-80.0, -71.5)
    ax.set_ylim(40.4, 45.1)
    
    # Subtle lat/lon grid lines so we have geographical reference
    ax.grid(color='#444444', linestyle='--', linewidth=0.5, alpha=0.5, zorder=1)
    
    plt.title(f"NYS Synoptic QPE Evolution\n{ts.strftime('%Y-%m-%d %H:%00')} UTC", 
              fontsize=22, color='white', pad=15)
    
    # Save Frame
    frame_name = f"temp_qpe_{ts.strftime('%Y%m%d_%H%M')}.png"
    plt.savefig(frame_name, dpi=150, facecolor='#2b2b2b', bbox_inches='tight')
    plt.close(fig) 
    
    frame_filenames.append(frame_name)

# --- 4. STITCH INTO GIF ---
print("\nStitching frames into a GIF...")
gif_path = "pure_qpe_evolution_jan2026.gif"

with imageio.get_writer(gif_path, mode='I', duration=500, loop=0) as writer:
    for filename in frame_filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

for filename in frame_filenames:
    os.remove(filename)

print(f"Success! Movie saved as: {gif_path}")